# Manhattan Café Wars: Starbucks vs 1,200 Competitors — Spatial Analysis with OSMnx

*Walk-distance analysis, subway ridership overlay, and density mapping with 100 % open data*

---

**What you'll get from this notebook:**
1. A **copy-paste OSMnx spatial analysis template** you can apply to any city
2. An **interactive map** of Manhattan's 1,300+ cafés × 123 subway stations
3. A **walk-distance demo** showing why straight-line distance lies to you

**Dataset:** [Manhattan Café Wars: Starbucks & Subway](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)

**Data sources:** OpenStreetMap (ODbL 1.0) · MTA via NY Open Data

## Section 0 — Setup & Data Loading

We load three CSV files from the companion dataset: 171 Starbucks locations, 1,335 cafés (all brands combined), and 123 subway station complexes with daily ridership. All coordinates are WGS84 (EPSG:4326).

The path auto-detects whether you're running on Kaggle or locally — no edits needed.

In [ ]:
# Kaggle environment: install spatial libraries (takes ~3 min)
!pip install -q osmnx folium mapclassify scikit-learn

# Verify osmnx installed
try:
    import osmnx
    print(f"osmnx {osmnx.__version__} installed successfully")
    OSMNX_AVAILABLE = True
except ImportError:
    print("WARNING: osmnx failed to install. Section 4 (walk-distance demo) will be skipped.")
    OSMNX_AVAILABLE = False

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from pathlib import Path
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# ----- data path (auto-detect Kaggle vs local) -----
# Kaggle mounts datasets at /kaggle/input/datasets/<user>/<dataset>/
KAGGLE_PATHS = [
    Path("/kaggle/input/manhattan-cafe-wars"),
    Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
]
LOCAL_DIR = Path("../../data/processed")  # relative to this notebook

DATA_DIR = None
ON_KAGGLE = False
for p in KAGGLE_PATHS:
    if p.exists():
        DATA_DIR = p
        ON_KAGGLE = True
        print(f"Running on Kaggle — data from {p}")
        break
if DATA_DIR is None and LOCAL_DIR.exists():
    DATA_DIR = LOCAL_DIR
    print(f"Running locally — data from {DATA_DIR.resolve()}")
if DATA_DIR is None:
    raise FileNotFoundError(
        "Data not found. On Kaggle, add the dataset 'shiratoriseto/manhattan-cafe-wars' "
        "via Add Data. Locally, ensure data/processed/ contains the CSV files."
    )

def show_map(m):
    """Display folium map — works on both Kaggle and local Jupyter."""
    if ON_KAGGLE:
        display(HTML(m._repr_html_()))
    else:
        display(m)

starbucks = pd.read_csv(DATA_DIR / "manhattan_starbucks_osm.csv")
cafes     = pd.read_csv(DATA_DIR / "manhattan_cafes_osm.csv")
stations  = pd.read_csv(DATA_DIR / "manhattan_mta_ridership_summary.csv")

# Sanity check
print(f"\nStarbucks : {len(starbucks):,} stores")
print(f"All cafés : {len(cafes):,} (incl. Starbucks, Dunkin', independents)")
print(f"Subway stn: {len(stations):,} station complexes")
print(f"CRS       : EPSG:4326 (WGS84)")
assert len(starbucks) == 171, f"Expected 171 Starbucks, got {len(starbucks)}"
assert len(cafes) == 1335,    f"Expected 1335 cafés, got {len(cafes)}"
assert len(stations) == 123,  f"Expected 123 stations, got {len(stations)}"

In [ ]:
# Quick look at the brand breakdown
print(cafes["brand_category"].value_counts().to_string())
print(f"\nTop 10 brands:")
print(cafes[cafes["brand"].notna()]["brand"].value_counts().head(10).to_string())

## Section 1 — The Map: Where Are They All?

Before any analysis, let's see the raw geography. This interactive map plots every café and subway station in Manhattan on four toggleable layers. The goal is a first visual answer to "where is the competition densest?" — and to see how tightly cafés cluster around transit.

- **Red** = Starbucks (171) · **Orange** = Dunkin' (115) · **Grey clusters** = independents & other brands (1,049) · **Blue** = subway stations (123)

Toggle layers on/off with the control in the top-right corner. Hover over any marker for details.

In [ ]:
# ---- Interactive folium map ----
MHT_CENTER = [40.7580, -73.9855]

m = folium.Map(location=MHT_CENTER, zoom_start=12, tiles="CartoDB positron")

# Layer 1: Starbucks (red)
fg_sbux = folium.FeatureGroup(name=f"Starbucks ({len(starbucks)})", show=True)
for _, r in starbucks.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=5, color="#D62828", fill=True, fill_opacity=0.8,
        tooltip=r["name"],
    ).add_to(fg_sbux)
fg_sbux.add_to(m)

# Layer 2: Dunkin' (orange)
dunkin = cafes[cafes["brand_category"] == "dunkin"]
fg_dunk = folium.FeatureGroup(name=f"Dunkin' ({len(dunkin)})", show=True)
for _, r in dunkin.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=5, color="#FF6600", fill=True, fill_opacity=0.8,
        tooltip=r["name"],
    ).add_to(fg_dunk)
fg_dunk.add_to(m)

# Layer 3: Other cafés (grey, clustered to handle 1,000+ points)
others = cafes[~cafes["brand_category"].isin(["starbucks", "dunkin"])]
fg_other = folium.FeatureGroup(name=f"Other cafés ({len(others)})", show=True)
mc = MarkerCluster()
for _, r in others.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=3, color="#888888", fill=True, fill_opacity=0.5,
        tooltip=r["name"],
    ).add_to(mc)
mc.add_to(fg_other)
fg_other.add_to(m)

# Layer 4: Subway stations (blue)
fg_stn = folium.FeatureGroup(name=f"Subway stations ({len(stations)})", show=True)
for _, r in stations.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=4, color="#0039A6", fill=True, fill_opacity=0.7,
        tooltip=f"{r['station_name']} — {r['avg_daily_ridership']:,.0f}/day",
    ).add_to(fg_stn)
fg_stn.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
show_map(m)

## Section 2 — Density: Who Dominates Where?

The map gives an impression, but impressions can mislead. This section quantifies the café landscape: how many stores does each brand category have, which Manhattan neighborhoods are most saturated, and where do Starbucks and its competitors concentrate?

We split Manhattan into seven latitude-band zones (Upper Manhattan through Financial District) and count stores by brand category in each.

In [ ]:
# ---- Brand breakdown bar chart ----
brand_counts = cafes["brand_category"].value_counts()

fig = px.bar(
    x=brand_counts.index.map({
        "independent": "Independent",
        "branded": "Other Branded",
        "starbucks": "Starbucks",
        "dunkin": "Dunkin'",
    }),
    y=brand_counts.values,
    color=brand_counts.index,
    color_discrete_map={
        "starbucks": "#D62828",
        "dunkin": "#FF6600",
        "branded": "#6B8EC2",
        "independent": "#AAAAAA",
    },
    labels={"x": "", "y": "Number of cafés"},
    title="Manhattan Café Landscape: 1,335 Locations by Category",
)
fig.update_layout(showlegend=False, template="plotly_white", height=400)
fig.show()

In [ ]:
# ---- Area-based density analysis ----
# Divide Manhattan into rough zones by latitude bands
def classify_zone(lat):
    if lat >= 40.800:
        return "Upper Manhattan (above 110th)"
    elif lat >= 40.774:
        return "Upper West/East Side"
    elif lat >= 40.754:
        return "Midtown (42nd-59th)"
    elif lat >= 40.735:
        return "Chelsea / Gramercy"
    elif lat >= 40.720:
        return "Greenwich / East Village"
    elif lat >= 40.710:
        return "SoHo / Lower East Side"
    else:
        return "Financial District"

cafes["zone"] = cafes["lat"].apply(classify_zone)

zone_brand = cafes.groupby(["zone", "brand_category"]).size().unstack(fill_value=0)
zone_brand = zone_brand.reindex(columns=["starbucks", "dunkin", "branded", "independent"])
zone_brand.columns = ["Starbucks", "Dunkin'", "Other Branded", "Independent"]
# Sort north → south
zone_order = [
    "Upper Manhattan (above 110th)",
    "Upper West/East Side",
    "Midtown (42nd-59th)",
    "Chelsea / Gramercy",
    "Greenwich / East Village",
    "SoHo / Lower East Side",
    "Financial District",
]
zone_brand = zone_brand.reindex(zone_order)

fig = px.bar(
    zone_brand,
    barmode="stack",
    color_discrete_map={
        "Starbucks": "#D62828",
        "Dunkin'": "#FF6600",
        "Other Branded": "#6B8EC2",
        "Independent": "#AAAAAA",
    },
    labels={"value": "Number of cafés", "zone": ""},
    title="Café Density by Manhattan Zone (north → south)",
)
fig.update_layout(template="plotly_white", height=450, legend_title_text="")
fig.show()

In [ ]:
# ---- Density heatmap: Starbucks vs Competitors side by side ----
sbux_pts = cafes[cafes["brand_category"] == "starbucks"]
comp_pts = cafes[cafes["brand_category"] != "starbucks"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Starbucks Density", "Competitor Density"),
    specs=[[{"type": "mapbox"}, {"type": "mapbox"}]],
    horizontal_spacing=0.02,
)

fig.add_trace(go.Densitymapbox(
    lat=sbux_pts["lat"], lon=sbux_pts["lon"],
    radius=20, colorscale="Reds", showscale=False,
    hoverinfo="skip",
), row=1, col=1)

fig.add_trace(go.Densitymapbox(
    lat=comp_pts["lat"], lon=comp_pts["lon"],
    radius=20, colorscale="Oranges", showscale=False,
    hoverinfo="skip",
), row=1, col=2)

fig.update_layout(
    height=500, template="plotly_white",
    mapbox=dict(style="carto-positron", center=dict(lat=40.758, lon=-73.985), zoom=11),
    mapbox2=dict(style="carto-positron", center=dict(lat=40.758, lon=-73.985), zoom=11),
    margin=dict(l=0, r=0, t=40, b=0),
    title_text="Where Is the Battle Fiercest?",
)
fig.show()

**Observations:**
- **Midtown** is the epicenter — both Starbucks and competitors pile up around Times Square, Grand Central, and Penn Station.
- **Upper Manhattan** has fewer cafés overall, but Starbucks has proportionally stronger presence there vs. independents.
- **Greenwich Village / East Village** is dominated by independents — this is artisanal coffee territory.
- The stacked bar chart shows that Starbucks commands **~13%** of all Manhattan café locations — far more than any single competitor.

## Section 3 — Distance Analysis: Cannibalization, Competition & Transit

Density maps show *where* stores cluster, but they don't answer *how close* individual stores are to each other. This section computes three straight-line distance metrics for every Starbucks location using `scipy.spatial.cKDTree`:

1. **Nearest Starbucks** — How close is the nearest sibling store? (cannibalization risk)
2. **Nearest Competitor** — How close is the nearest non-Starbucks café? (competitive pressure)
3. **Nearest Subway Station** — Is Starbucks systematically chasing commuter foot traffic?

All distances are Euclidean (straight-line), computed via a simple planar approximation valid for Manhattan's scale (error < 0.5%). Section 4 will show why real walking distances are longer.

In [ ]:
# ---- Distance computation helper ----
# CRS: EPSG:4326 → approximate meters using simple planar projection
REF_LAT = 40.75  # Manhattan reference latitude
M_PER_DEG_LAT = 111_320
M_PER_DEG_LON = M_PER_DEG_LAT * np.cos(np.radians(REF_LAT))

def latlon_to_meters(lat, lon):
    """Convert lat/lon arrays to approximate meter coordinates."""
    return lat * M_PER_DEG_LAT, lon * M_PER_DEG_LON

# ---- 1. Nearest Starbucks-to-Starbucks (self-cannibalization) ----
sbux_y, sbux_x = latlon_to_meters(starbucks["lat"].values, starbucks["lon"].values)
sbux_coords = np.column_stack([sbux_x, sbux_y])
sbux_tree = cKDTree(sbux_coords)

# k=2: first match is itself (dist=0), second is the nearest *other* Starbucks
dd_sbux, ii_sbux = sbux_tree.query(sbux_coords, k=2)
starbucks["nearest_starbucks_dist_m"] = dd_sbux[:, 1]
starbucks["nearest_starbucks_name"] = starbucks.iloc[ii_sbux[:, 1]]["name"].values

# ---- 2. Nearest competitor (non-Starbucks café) ----
competitors = cafes[cafes["brand_category"] != "starbucks"]
comp_y, comp_x = latlon_to_meters(competitors["lat"].values, competitors["lon"].values)
comp_tree = cKDTree(np.column_stack([comp_x, comp_y]))

dd_comp, ii_comp = comp_tree.query(sbux_coords)
starbucks["nearest_competitor_dist_m"] = dd_comp
starbucks["nearest_competitor_name"] = competitors.iloc[ii_comp]["name"].values

# ---- 3. Nearest subway station ----
stn_y, stn_x = latlon_to_meters(stations["lat"].values, stations["lon"].values)
stn_tree = cKDTree(np.column_stack([stn_x, stn_y]))

dd_stn, ii_stn = stn_tree.query(sbux_coords)
starbucks["nearest_station_dist_m"] = dd_stn
starbucks["nearest_station_name"] = stations.iloc[ii_stn]["station_name"].values

# Summary
print("Distance summary (straight-line, meters):")
print(f"{'':30s} {'Median':>8s} {'Mean':>8s} {'Min':>8s} {'Max':>8s}")
for col, label in [
    ("nearest_starbucks_dist_m",  "Starbucks → nearest Starbucks"),
    ("nearest_competitor_dist_m", "Starbucks → nearest competitor"),
    ("nearest_station_dist_m",    "Starbucks → nearest subway stn"),
]:
    s = starbucks[col]
    print(f"{label:30s} {s.median():8.0f} {s.mean():8.0f} {s.min():8.0f} {s.max():8.0f}")

In [ ]:
# ---- Three distance histograms (plotly subplots) ----
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Nearest Starbucks<br>(cannibalization)",
        "Nearest Competitor",
        "Nearest Subway Station",
    ),
    horizontal_spacing=0.08,
)

# 1) Starbucks → nearest Starbucks
fig.add_trace(go.Histogram(
    x=starbucks["nearest_starbucks_dist_m"],
    nbinsx=25, marker_color="#D62828", opacity=0.8,
    name="→ Starbucks", showlegend=False,
), row=1, col=1)

# 2) Starbucks → nearest competitor
fig.add_trace(go.Histogram(
    x=starbucks["nearest_competitor_dist_m"],
    nbinsx=25, marker_color="#FF6600", opacity=0.8,
    name="→ Competitor", showlegend=False,
), row=1, col=2)

# 3) Starbucks → nearest subway station
fig.add_trace(go.Histogram(
    x=starbucks["nearest_station_dist_m"],
    nbinsx=25, marker_color="#0039A6", opacity=0.8,
    name="→ Subway", showlegend=False,
), row=1, col=3)

# 500m reference line on subway chart
fig.add_vline(x=500, line_dash="dash", line_color="red",
              annotation_text="500m", row=1, col=3)

fig.update_xaxes(title_text="Distance (m)")
fig.update_yaxes(title_text="Count", col=1)
fig.update_layout(
    title_text="How Close Is the Nearest…? (171 Starbucks stores, straight-line distance)",
    template="plotly_white",
    height=400,
    margin=dict(t=80),
)
fig.show()

In [ ]:
# ---- Closest pairs: which Starbucks stores are dangerously close? ----
closest_pairs = starbucks.nsmallest(10, "nearest_starbucks_dist_m")[
    ["name", "nearest_starbucks_name", "nearest_starbucks_dist_m",
     "nearest_competitor_name", "nearest_competitor_dist_m",
     "nearest_station_name", "nearest_station_dist_m"]
].copy()
closest_pairs.columns = [
    "Starbucks", "Nearest Starbucks", "Dist (m)",
    "Nearest Competitor", "Comp Dist (m)",
    "Nearest Station", "Stn Dist (m)",
]
closest_pairs["Dist (m)"] = closest_pairs["Dist (m)"].round(0).astype(int)
closest_pairs["Comp Dist (m)"] = closest_pairs["Comp Dist (m)"].round(0).astype(int)
closest_pairs["Stn Dist (m)"] = closest_pairs["Stn Dist (m)"].round(0).astype(int)
closest_pairs.reset_index(drop=True)

**Key takeaways:**

- **Cannibalization is real:** The median distance between a Starbucks and its nearest sibling is remarkably short. Some pairs are within a single block of each other. This is the signature of a "dominant" location strategy — saturate an area before competitors can.
- **Competitors are even closer:** For most Starbucks, the nearest competitor (Dunkin', independent, etc.) is closer than the nearest Starbucks. The battle for every corner is intense.
- **Subway proximity is nearly universal:** The vast majority of Starbucks stores sit within 500m of a subway station, confirming that commuter foot traffic drives location decisions.

But all these distances are **straight-line**. In a city of blocks and avenues, the real walk is 20-40% longer. Section 4 shows exactly how much.

## Section 4 — OSMnx Walk-Distance Demo: Straight Lines Lie

Section 3 showed that many Starbucks stores are just a few hundred meters from each other — in straight-line distance. But Manhattan is a grid of blocks, not open fields. [OSMnx](https://osmnx.readthedocs.io/) lets us download the actual street network from OpenStreetMap and compute the **real walking distance** between two points.

Here we pick one Starbucks and measure the walk to its nearest sibling. The goal is not to analyze all 171 stores (that's Part 2) — it's to show **why straight-line distance underestimates reality** and why this tool matters.

> **Note:** This cell downloads a small street network (~1 km²) from the Overpass API. On Kaggle, toggle **Internet → ON** in Settings.

In [ ]:
if not OSMNX_AVAILABLE:
    print("Skipping Section 4 — osmnx not available in this environment.")
    print("To run this section, install osmnx locally: pip install osmnx")
else:
    import osmnx as ox

    # Pick a Starbucks pair ~200-300m apart for a clear walk-vs-straight demo
    # Very close pairs (<100m) suffer from node-snapping artifacts;
    # we want a pair where the grid detour is visually obvious
    candidates = starbucks[
        starbucks["nearest_starbucks_dist_m"].between(150, 350)
    ].copy()
    # Pick one near Midtown for a recognizable map
    candidates["midtown_dist"] = (
        (candidates["lat"] - 40.755).abs() + (candidates["lon"] + 73.985).abs()
    )
    demo_idx = candidates["midtown_dist"].idxmin()
    demo_sbux = starbucks.loc[demo_idx]

    # Find its nearest Starbucks neighbor
    nearest_idx = ii_sbux[demo_idx, 1]  # from the k=2 KDTree query in Section 3
    demo_neighbor = starbucks.iloc[nearest_idx]

    straight_dist = starbucks.loc[demo_idx, "nearest_starbucks_dist_m"]

    print(f"Store A   : {demo_sbux['name']}")
    print(f"  at      : ({demo_sbux['lat']:.5f}, {demo_sbux['lon']:.5f})")
    print(f"Store B   : {demo_neighbor['name']}")
    print(f"  at      : ({demo_neighbor['lat']:.5f}, {demo_neighbor['lon']:.5f})")
    print(f"Straight  : {straight_dist:.0f}m")

In [ ]:
if OSMNX_AVAILABLE:
    # ---- Download walk network & compute shortest path ----
    midpoint = ((demo_sbux["lat"] + demo_neighbor["lat"]) / 2,
                (demo_sbux["lon"] + demo_neighbor["lon"]) / 2)

    # Download a small walking network around the two stores
    G = ox.graph_from_point(midpoint, dist=800, network_type="walk")

    # Snap stores to nearest network nodes
    orig = ox.nearest_nodes(G, demo_sbux["lon"], demo_sbux["lat"])
    dest = ox.nearest_nodes(G, demo_neighbor["lon"], demo_neighbor["lat"])

    # Shortest walk path
    route = ox.shortest_path(G, orig, dest, weight="length")

    # Calculate walk distance along the route
    edge_lengths = ox.routing.route_to_gdf(G, route)["length"]
    walk_dist = edge_lengths.sum()

    ratio = walk_dist / straight_dist
    print(f"Straight-line distance : {straight_dist:.0f}m")
    print(f"Walk distance          : {walk_dist:.0f}m")
    print(f"Ratio (walk/straight)  : {ratio:.2f}×")
    print(f"\nThe walk is {ratio - 1:.0%} longer than the crow flies.")

In [ ]:
if OSMNX_AVAILABLE:
    # ---- Plot the walk route vs straight line on a folium map ----
    route_map = folium.Map(location=list(midpoint), zoom_start=17, tiles="CartoDB positron")

    # Walk route (red solid line)
    route_coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in route]
    folium.PolyLine(
        route_coords, color="#E63946", weight=5, opacity=0.8,
        tooltip=f"Walk: {walk_dist:.0f}m",
    ).add_to(route_map)

    # Straight-line (blue dashed)
    folium.PolyLine(
        [(demo_sbux["lat"], demo_sbux["lon"]),
         (demo_neighbor["lat"], demo_neighbor["lon"])],
        color="#457B9D", weight=3, dash_array="10",
        tooltip=f"Straight: {straight_dist:.0f}m",
    ).add_to(route_map)

    # Store A marker (red)
    folium.Marker(
        [demo_sbux["lat"], demo_sbux["lon"]],
        icon=folium.Icon(color="red", icon="coffee", prefix="fa"),
        tooltip=f"A: {demo_sbux['name']}",
    ).add_to(route_map)

    # Store B marker (dark red)
    folium.Marker(
        [demo_neighbor["lat"], demo_neighbor["lon"]],
        icon=folium.Icon(color="darkred", icon="coffee", prefix="fa"),
        tooltip=f"B: {demo_neighbor['name']}",
    ).add_to(route_map)

    show_map(route_map)

The **red line** is the actual walking route along Manhattan's streets; the **blue dashed line** is the straight-line distance. In the grid, you always walk around blocks — adding 20-40% to the real distance.

This single example demonstrates a critical point: **straight-line distance systematically underestimates how far apart two locations really are for a pedestrian.** When Starbucks decides whether two stores cannibalize each other, or whether a location captures subway foot traffic, the walk distance is what matters — not the crow-fly number.

In **Part 2** of this series, we'll compute walk distances for all 171 stores and use them for cannibalization scoring and demand modeling. This notebook just shows the tool — OSMnx — and the concept.

> **Try this for your city:** change the coordinates in `ox.graph_from_point()`. OSMnx works anywhere OpenStreetMap has data — which is most of the world.

## Section 5 — What's Next: The Series Roadmap

This notebook scratched the surface — a first look at Manhattan's 1,300+ cafés, basic distance metrics, and one OSMnx demo. The real analysis comes next.

**This series will cover:**
- **Spatial autocorrelation** (Moran's I) to detect statistically significant clusters of Starbucks stores
- **Demand proxy modeling** using subway ridership × building attributes (floor area, land use, assessed value) to score every block in Manhattan
- **Walk-distance cannibalization analysis** — computing OSMnx network distances for all 171 stores to quantify self-competition
- **10-K annual report NLP** — tracking how Starbucks' corporate language around "growth", "saturation", and "store optimization" shifted over 30 years of SEC filings

### Part 1 — 30 Years of Starbucks Expansion
A US-wide animated map of Starbucks growth from 1 store (1971) to 16,000+, paired with NLP analysis of 10-K filing language to show how corporate strategy evolved alongside the store count.

### Part 2 — Manhattan Deep-Dive: The Dominant Strategy
Cannibalization scoring, demand surface modeling, optimal store placement backtesting — all using the open data introduced in this notebook plus Census/ACS demographics and MapPLUTO building data.

All analysis uses **100% open data**: OpenStreetMap, SEC EDGAR, US Census, MTA, NYC Open Data.

---

### References & Licenses

- **OpenStreetMap data** (cafés & Starbucks locations): © OpenStreetMap contributors, available under the [Open Database License (ODbL) v1.0](https://opendatacommons.org/licenses/odbl/1-0/)
- **MTA Subway Ridership**: Metropolitan Transportation Authority via [New York State Open Data](https://data.ny.gov), OPEN NY Terms of Use
- **OSMnx**: Boeing, G. (2017). OSMnx: New methods for acquiring, constructing, analyzing, and visualizing complex street networks. *Computers, Environment and Urban Systems*, 65, 126-139.

---
*Built with pandas · plotly · folium · OSMnx · scipy*